# Hakam — training v4

Round 3 curves: only VideoMAE-base lowered validation loss, every run peaked at epoch 3–5, and training used one view per action out of 2–4. This round uses every view, one backbone with a head per task (card, offence, action class, body part), weight averaging and a 3-seed ensemble. The test split is scored once, at the end.

1. **Runtime → Change runtime type → L4 or A100 GPU**
2. **Runtime → Run all** (about 1 h 45 min on L4, 50 min on A100)

## 1. GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU - change the runtime type."
print(torch.cuda.get_device_name(0))
!df -h /content | tail -1

## 2. Code from GitHub, data and labels from Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!test -d /content/hakam && (cd /content/hakam && git pull -q) || git clone -q https://github.com/FerasMad/hakam.git /content/hakam
!cd /content/hakam && python scripts/colab_setup.py

## 3. Install and cache every view

In [ ]:
%cd /content/hakam
!pip install -q transformers
!python scripts/round4.py cache

## 4. mv9 — every view, card only
Same model as Exp 7, but all views in training and averaged per action at evaluation.

In [ ]:
!python scripts/round4.py mv9

## 5. mv10 — one backbone, four heads
Adds offence, action class and body part heads, borderline cards as 0.5 targets, and weight averaging (EMA).

In [ ]:
!python scripts/round4.py mv10

## 6. mv11 — motion-pretrained backbone
The same recipe on VideoMAE fine-tuned on Something-Something v2.

In [ ]:
!python scripts/round4.py mv11

## 7. mv12 — two more seeds of the winner

In [ ]:
!python scripts/round4.py mv12

## 8. Ensemble, thresholds from valid, test once

In [ ]:
!python scripts/round4.py final

## 9. Results, figures and save to Drive

In [ ]:
!python scripts/summarise_runs.py --save /content/drive/MyDrive/hakam_colab/runs